<center>

## <h2><strong><font color="navy">SENTIMENT ANALYSIS</font></strong></h2>

</center>

## Pendahuluan

Ulasan pengguna merupakan aspek penting dalam pengembangan dan peningkatan kualitas aplikasi, terutama aplikasi layanan on-demand seperti Gojek. Melalui analisis sentimen terhadap ulasan pengguna, perusahaan dapat memahami persepsi, kepuasan, maupun keluhan yang dialami pengguna, serta menentukan area layanan yang perlu diperbaiki. analisis ini bertujuan untuk melakukan analisis sentimen pada ulasan aplikasi Gojek guna mendapatkan wawasan mengenai kepuasan pengguna secara mendalam.

Dataset yang digunakan dalam analisis ini adalah [Gojek App Reviews in Bahasa Indonesia](https://www.kaggle.com/datasets/ucupsedaya/gojek-app-reviews-bahasa-indonesia/). Dataset ini mencakup berbagai ulasan pengguna aplikasi Gojek yang disertai dengan skor penilaian, versi aplikasi, dan waktu pemberian ulasan. Dengan memanfaatkan data ini, analisis akan menerapkan metode pemrosesan bahasa alami (Natural Language Processing/NLP) untuk mengklasifikasikan sentimen ulasan menjadi positif, negatif, atau netral, yang kemudian hasilnya dapat digunakan untuk mengevaluasi dan meningkatkan kualitas layanan aplikasi Gojek.

## Load Data

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('/content/GojekAppReviewV4.0.0-V4.9.3_Cleaned.csv')
df.head()

## EDA & Preprocessing

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
sum(df['appVersion'].str.startswith("4.8"))

Terdapat sebanyak 8091 ulasan pengguna yang menggunakan versi aplikasi Gojek dengan awalan versi "4.8".

In [ ]:
# ambil kolom yg dibutuhkan
df = df[df['appVersion'].str.startswith("4.8")]
df = df.loc[:, ['userName', 'content', 'score']]

df.head()

In [ ]:
!pip install nltk
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')

# tokenization
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string

# hapus duplikasi
df = df.dropna(subset=['content']).drop_duplicates()

# stopwords
stop_words = stopwords.words('indonesian') + stopwords.words('english') + ["yg", "gak", "ngisi", "udah", "d", "sih", "nya", "srg", "utk", "byk", "gk", "ga", "aja", "tp", "udh"]
df['content'] = df['content'].apply(lambda x: [word.lower() for word in word_tokenize(x) if (word.isalpha() and word.lower() not in stop_words)])

# normalisasi teks
df['content'] = df['content'].apply(lambda x: ' '.join(x))

df.head()

**Sastrawi** digunakan untuk preprocessing teks bahasa Indonesia seperti stemming atau normalisasi kata, sedangkan **VaderSentiment** digunakan untuk analisis sentimen secara otomatis dari teks, terutama dalam konteks media sosial.

In [ ]:
!pip install Sastrawi
!pip install VaderSentiment

In [ ]:
# stemming
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

stemmer = StemmerFactory().create_stemmer()
df['content'] = df['content'].apply(lambda x: ' '.join([stemmer.stem(word) for word in x.split()]))

df.head(5)

In [ ]:
# labelling
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

additional_lexicon_id = {
    'kecewa': -0.4,
    'rugi': -1,
    'buruk': -0.6,
    'jelek': -0.6,
    'lelet': -0.7,
    'gagal': -0.5,
    'parah': -0.6,
    'mahal': -0.3,
    'tolong': -0.1,
    'hilang': -0.3,
    'gajelas': -0.3,
    'gj': -0.3,
    'promo': 0.6,
    'kadang': -0.1,
    'maling': -0.5,
    'ganggu': 0.3,
    'sedot': -0.5,
    'bagus': 0.5,
    'pulsa': 0,
    'potong': -1,
    'baik': 0.5,
    'kntl': -1,
    'ngelag': -0.8,
    'salah': -0.5,
    'bintang': 0,
    'benerin': -0.4,
    'lambat': -0.8,
    'siput': -0.4,
    'mati': -0.7,
    'minimal': -0.3,
    'susah': -0.6,
    'nagih': -0.6,
    'capek': -0.7,
    'kacau': -0.3,
    'tagih': -0.3,
    'mantap': 1,
    'puas': 0.9,
    'sampah': -0.5,
    'sulit': -0.6,
    'aneh': -0.4,
}

analyzer.lexicon.update(additional_lexicon_id)

df['sentimen'] = df['content'].apply(lambda x: 'Positif' if analyzer.polarity_scores(x)['compound'] > 0 else ('Negatif' if analyzer.polarity_scores(x)['compound'] < 0 else 'Netral'))

df

In [ ]:
# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df['content'])

### Analisis Sentimen

In [ ]:
from wordcloud import WordCloud
from plotly import graph_objs as go
import plotly.express as px
import plotly.figure_factory as ff
from collections import Counter

#### WordCloud

In [ ]:
df_netral = df[df['sentimen'] == 'Netral']
all_words_netral = ' '.join([twts for twts in df_netral['content']])
wordcloud_netral = WordCloud(width=500, height=300, random_state=21, max_font_size=110).generate(all_words_netral)

plt.imshow(wordcloud_netral, interpolation="bilinear")
# plt.axis('off')
plt.title('Word Cloud dari Sentimen Netral')
plt.show()

Word cloud sentimen netral menunjukkan bahwa pengguna aplikasi Gojek umumnya menyoroti kata-kata seperti "gojek," "bantu," dan "aplikasi," yang mengindikasikan fungsi dasar layanan aplikasi yang dianggap membantu oleh pengguna. Kata lainnya seperti "driver," "cepat," dan "mudah" juga muncul, mencerminkan aspek kecepatan dan kemudahan layanan yang diberikan meski dengan sentimen netral.

In [ ]:
df_positif = df[df['sentimen'] == 'Positif']
all_words_positif = ' '.join([twts for twts in df_positif['content']])
wordcloud_positif = WordCloud(width=500, height=300, random_state=21, max_font_size=110).generate(all_words_positif)

plt.imshow(wordcloud_positif, interpolation="bilinear")
# plt.axis('off')
plt.title('Word Cloud dari Sentimen Positif')
plt.show()

Word cloud sentimen positif menunjukkan bahwa pengguna aplikasi Gojek sering menggunakan kata seperti "bagus," "baik," "mantap," dan "promo," mengindikasikan bahwa pengguna merasa puas dengan kualitas layanan, performa aplikasi, dan promosi yang diberikan. Kata "good" dan "driver" juga muncul secara signifikan, mencerminkan apresiasi terhadap pelayanan yang baik dari mitra pengemudi serta pengalaman pengguna secara keseluruhan yang menyenangkan.

In [ ]:
df_negatif = df[df['sentimen'] == 'Negatif']
all_words_negatif = ' '.join([twts for twts in df_negatif['content']])
wordcloud_negatif = WordCloud(width=500, height=300, random_state=21, max_font_size=110).generate(all_words_negatif)

plt.imshow(wordcloud_negatif, interpolation="bilinear")
# plt.axis('off')
plt.title('Word Cloud dari Sentimen Negatif')
plt.show()

Word cloud sentimen negatif menunjukkan bahwa pengguna aplikasi Gojek sering menggunakan kata "driver," "susah," "kecewa," dan "parah," mencerminkan ketidakpuasan pengguna terhadap kualitas layanan, terutama yang terkait dengan kinerja mitra pengemudi. Kata seperti "cancel," "mahal," dan "error" juga muncul, menunjukkan adanya keluhan terkait masalah pembatalan pesanan, harga layanan, serta gangguan teknis dalam aplikasi.

#### Distribusi Target

In [ ]:
temp = df.groupby('sentimen').count()['content'].reset_index().sort_values(by='content',ascending=False)
temp.style.background_gradient(cmap='inferno_r')

In [ ]:
plt.figure(figsize=(12,6))
sns.countplot(x='sentimen',data=df)

Bar chart menunjukkan bahwa ulasan dengan sentimen netral memiliki jumlah paling tinggi, diikuti oleh sentimen positif, dan terakhir sentimen negatif.

In [ ]:
fig = go.Figure(go.Funnelarea(
    text =temp.sentimen,
    values = temp.content,
    title = {"position": "top center", "text": "Funnel-Chart dari Distribusi target"}
    ))
fig.show()

 Funnel-chart menegaskan distribusi tersebut dengan persentase 49,3% netral, 31,4% positif, dan 19,3% negatif, menunjukkan bahwa sebagian besar pengguna memiliki pandangan netral terhadap aplikasi Gojek.

Library `palettable` (dalam hal ini, Pastel1_7) digunakan untuk membuat visualisasi menjadi lebih menarik dan mudah dibedakan.

In [ ]:
!pip install palettable
from palettable.colorbrewer.qualitative import Pastel1_7

In [ ]:
unique_netral_words = df_netral['content'].str.split(expand=True).stack().value_counts().reset_index()
unique_netral_words.columns = ['words', 'count']
top_20_words = unique_netral_words.head(12)
plt.figure(figsize=(12, 6))
my_circle = plt.Circle((0, 0), 0.7, color='white')
plt.pie(top_20_words['count'], labels=top_20_words['words'], colors=Pastel1_7.hex_colors)
plt.gca().add_artist(my_circle)
plt.title('Donut Plot dari Sentimen Netral')
plt.show()

Grafik sentimen netral menunjukkan bahwa pengguna paling sering menggunakan kata "bantu," "aplikasi," dan "gojek," menunjukkan bahwa pengguna secara umum merasa layanan Gojek membantu meski tidak memiliki sentimen khusus. Kata "driver," "mudah," dan "pesan" juga muncul, menegaskan bahwa pengguna merasa netral terhadap kemudahan penggunaan aplikasi serta interaksi dengan mitra pengemudi.

In [ ]:
unique_positif_words = df_positif['content'].str.split(expand=True).stack().value_counts().reset_index()
unique_positif_words.columns = ['words', 'count']
top_20_words = unique_positif_words.head(12)
plt.figure(figsize=(12, 6))
my_circle = plt.Circle((0, 0), 0.7, color='white')
plt.pie(top_20_words['count'], labels=top_20_words['words'], colors=Pastel1_7.hex_colors)
plt.gca().add_artist(my_circle)
plt.title('Donut Plot dari Sentimen positif')
plt.show()

Grafik sentimen positif memperlihatkan kata "bagus," "mantap," dan "baik," mengindikasikan kepuasan pengguna terhadap kualitas layanan aplikasi Gojek secara umum. Kata "promo," "top," dan "good" yang juga sering disebutkan menunjukkan pengguna sangat menghargai adanya promosi serta layanan berkualitas dari mitra pengemudi.

In [ ]:
unique_negatif_words = df_negatif['content'].str.split(expand=True).stack().value_counts().reset_index()
unique_negatif_words.columns = ['words', 'count']
top_20_words = unique_negatif_words.head(12)
plt.figure(figsize=(12, 6))
my_circle = plt.Circle((0, 0), 0.7, color='white')
plt.pie(top_20_words['count'], labels=top_20_words['words'], colors=Pastel1_7.hex_colors)
plt.gca().add_artist(my_circle)
plt.title('Donut Plot dari Sentimen negatif')
plt.show()

Grafik sentimen negatif memperlihatkan dominasi kata seperti "gojek," "driver," dan "kecewa," yang menunjukkan ketidakpuasan pengguna terkait layanan yang diberikan, terutama terkait perilaku driver dan performa aplikasi. Kata "mahal," "susah," dan "saldo" juga cukup dominan, mencerminkan ketidaknyamanan pengguna terhadap biaya layanan serta berbagai kendala teknis maupun administratif dalam aplikasi Gojek.

#### SPLIT

Data displit menjadi 80% data train dan 20% data test.

In [ ]:
# splitting
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, df['sentimen'], test_size=0.2, random_state=42)
X_train.shape, X_test.shape

#### Resampling target

Diterapkan teknik SMOTE (Synthetic Minority Over-sampling Technique) untuk menyeimbangkan jumlah data pada setiap kelas sentimen,

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

plt.figure(figsize=(12, 6))
sns.countplot(x=y_train)
plt.title('Distribusi target untuk modeling')
plt.show()

Data sentimen positif, negatif, dan netral memiliki distribusi yang seimbang sebelum proses modeling dilakukan.

## Model

Akan digunakan model Random Forest dengan menentukan beberapa parameter seperti jumlah pohon (`n_estimators`), kedalaman pohon (`max_depth`), jumlah sampel minimum untuk membagi node (`min_samples_split`), serta jumlah sampel minimum pada setiap daun (`min_samples_leaf`), yang nantinya akan diuji dengan RandomizedSearchCV.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
# init parameters
rf_param_grid = {'n_estimators': [50, 100, 200],
                 'max_depth': [None, 10, 20, 30],
                 'min_samples_split': [2, 5, 10],
                 'min_samples_leaf': [1, 2, 4]}

### Random Forest

In [ ]:
rf_model = RandomizedSearchCV(RandomForestClassifier(random_state=42), rf_param_grid, n_iter=10, cv=5, scoring='accuracy', random_state=42)
rf_model.fit(X_train, y_train)

Hasil pencarian parameter terbaik menggunakan metode RandomizedSearchCV pada model Random Forest, dengan parameter terbaik yang ditemukan yaitu `min_samples_split=5`.

### Model Eval

In [ ]:
# print best param
print("\nBest Parameters for Random Forest:", rf_model.best_params_)

Parameter terbaik untuk model Random Forest menunjukkan bahwa performa optimal tercapai dengan menggunakan 100 pohon, minimal 5 sampel untuk pembagian node, minimal 1 sampel pada daun, dan tanpa batasan kedalaman pohon. Konfigurasi ini dipilih karena memberikan keseimbangan terbaik antara kompleksitas model dan akurasi prediksi.

In [ ]:
# evaluasi model
from sklearn.metrics import classification_report

y_pred_rf = rf_model.best_estimator_.predict(X_test)

print("\n\nClassification Report for Random Forest (Tuned):")
print(classification_report(y_test, y_pred_rf))

#### 1. Precision
- **Negatif (0.82)**:
  Dari seluruh ulasan yang diprediksi model sebagai sentimen negatif, 82%-nya memang benar-benar memiliki sentimen negatif terhadap aplikasi Gojek. Sekitar 18% sisanya merupakan kesalahan klasifikasi (seharusnya netral atau positif).

- **Netral (0.96)**:  
  Ulasan yang diprediksi sebagai netral oleh model memiliki tingkat ketepatan tinggi, yaitu 96%, yang berarti sebagian besar ulasan netral berhasil dikenali dengan akurat.

- **Positif (0.98)**:  
  Kategori ini menunjukkan performa paling baik, dengan 98% ulasan yang diprediksi sebagai positif memang benar-benar positif terhadap aplikasi Gojek.

#### 2. Recall
- **Negatif (0.92)**:  
  Sebanyak 92% ulasan yang benar-benar negatif berhasil dideteksi dengan baik oleh model, menunjukkan kemampuan yang baik dalam mengenali sentimen negatif dari pengguna.

- **Netral (0.96)**:  
  Model berhasil mengidentifikasi sebanyak 96% dari seluruh ulasan yang memiliki sentimen netral, menunjukkan kemampuan model dalam mengenali mayoritas ulasan netral secara efektif.

- **Positif (0.90)**:  
  Dari total ulasan yang positif, model berhasil mengenali 90% di antaranya, sedikit lebih rendah dibandingkan kategori lainnya, tetapi masih menunjukkan performa yang sangat baik.

#### 3. F1-Score
- **Negatif (0.87)**:  
  Skor F1 (gabungan dari precision dan recall) pada kategori negatif sebesar 0.87 menunjukkan keseimbangan yang baik antara precision dan recall dalam mengklasifikasikan ulasan negatif.

- **Netral (0.96)**:  
  F1-Score untuk sentimen netral sangat tinggi (0.96), menunjukkan bahwa model mampu secara konsisten dan akurat mengenali ulasan netral dengan tingkat kesalahan yang rendah.

- **Positif (0.94)**:  
  Kategori positif memiliki skor F1 sebesar 0.94, menunjukkan model cukup konsisten dalam memprediksi ulasan positif.

#### 4. Accuracy (0.93)  
Model secara keseluruhan memiliki akurasi sebesar **93%**, artinya dari total 1618 ulasan yang diprediksi, sekitar 93%-nya telah diklasifikasikan secara tepat oleh model, mengindikasikan performa keseluruhan yang tinggi dalam analisis sentimen aplikasi Gojek.

#### 5. Macro Average (Rata-rata tanpa memperhatikan jumlah data tiap kelas)
- Precision: **0.92**
- Recall: **0.93**
- F1-score: **0.92**

Nilai ini menunjukkan bahwa secara rata-rata, model memiliki performa tinggi yang merata untuk setiap kategori sentimen, tanpa terlalu terpengaruh oleh jumlah ulasan pada tiap kategori.

### 6. Weighted Average (Rata-rata dengan memperhatikan jumlah data tiap kelas)
- Precision: **0.94**
- Recall: **0.93**
- F1-score: **0.94**

Nilai ini menunjukkan performa model secara keseluruhan lebih condong ke kategori dengan data lebih banyak (netral), tetapi masih mempertahankan kualitas prediksi yang baik.